# LeetCode 149 — Max Points on a Line (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxPoints(self, points: List[List[int]]) -> int:

```

Bài này là **Hard** không phải vì thuật toán phức tạp, mà vì có rất nhiều **cái bẫy nhỏ**: chia cho 0 khi gặp đường thẳng đứng, số thực (float) làm tròn sai, cùng một hệ số góc nhưng viết theo nhiều kiểu (`1/2`, `2/4`, `-1/-2`)... Notebook này đi từ hình học lớp 7 (tọa độ, hệ số góc) lên tới lời giải chính xác tuyệt đối chỉ dùng **số nguyên**.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc và **vẽ** các điểm trên mặt phẳng tọa độ bằng ASCII để tự đếm tay.
2. **Tích chéo** (cross product) — công cụ kiểm tra 3 điểm thẳng hàng **không cần phép chia**.
3. **Cách 1 — Brute force** `O(n³)`: thử mọi cặp điểm làm "đường", đếm mọi điểm nằm trên đó bằng tích chéo.
4. **Cách 2 — Điểm neo + dict theo hệ số góc float** `O(n²)`: nhanh hơn nhưng vướng **chia cho 0** và **sai số float** (có ví dụ cụ thể).
5. **Cách 3 — Điểm neo + hệ số góc dạng phân số tối giản `(dy, dx)`** chuẩn hóa dấu bằng `math.gcd` — **cách chính**, chính xác tuyệt đối, `O(n² log C)`.
6. **Cách 4 — `fractions.Fraction`** và **Cách 5 — phương trình đường thẳng `ax + by = c`**: hai cách biểu diễn "chính xác" khác để mở rộng tư duy.
7. Vì sao đúng, so sánh các cách, lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Cho mảng `points` trong đó `points[i] = [xi, yi]` biểu diễn một **điểm** trên mặt phẳng X-Y. Hãy trả về **số điểm nhiều nhất cùng nằm trên một đường thẳng**.

**Example 1**

```text
Input:  points = [[1,1],[2,2],[3,3]]
Output: 3
```

**Example 2**

```text
Input:  points = [[1,1],[3,2],[5,3],[4,1],[2,3],[1,4]]
Output: 4
```

**Constraints**

- `1 <= points.length <= 300`
- `points[i].length == 2`
- `-10^4 <= xi, yi <= 10^4`
- **All the points are unique** (mọi điểm đều phân biệt — không có hai điểm trùng tọa độ).

> Ghi chú: phiên bản rất cũ của bài này dùng class `Point` và **cho phép điểm trùng nhau**. Đề hiện hành dùng `List[List[int]]` và các điểm **phân biệt**. Ta giải đúng đề hiện hành; phiên bản có điểm trùng được để làm bài tập 6.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

points1 = [[1, 1], [2, 2], [3, 3]]
points2 = [[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]

print("Example 1:", points1, "-> mong đợi 3")
print("Example 2:", points2, "-> mong đợi 4")
print("Số điểm của example 2:", len(points2))
print("Điểm index 3 của example 2:", points2[3], "-> x =", points2[3][0], ", y =", points2[3][1])

Example 1: [[1, 1], [2, 2], [3, 3]] -> mong đợi 3
Example 2: [[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]] -> mong đợi 4
Số điểm của example 2: 6
Điểm index 3 của example 2: [4, 1] -> x = 4 , y = 1


## 2. Vẽ các điểm ra cho dễ thấy

Mỗi điểm `[x, y]` nghĩa là: đi sang **phải** `x` bước (trục hoành, trục X), đi **lên** `y` bước (trục tung, trục Y).

Ta viết một hàm nhỏ vẽ lưới ASCII:

- `.` là ô trống,
- `o` là một điểm của input,
- `#` là điểm nằm trên đường thẳng mà ta muốn **tô sáng** (highlight),
- hàng trên cùng là `y` lớn nhất (giống trục Y trong vở toán: đi lên là tăng).

Hàm chỉ để **nhìn**, không phải một phần của lời giải.

In [2]:
def draw_points(points, highlight=()):
    """Vẽ điểm lên lưới ASCII. highlight = các index cần tô '#'."""
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    where = {(p[0], p[1]): i for i, p in enumerate(points)}   # tọa độ -> index
    for y in range(max(ys), min(ys) - 1, -1):                 # vẽ từ trên xuống
        row = []
        for x in range(min(xs), max(xs) + 1):
            if (x, y) not in where:
                row.append(".")
            elif where[(x, y)] in highlight:
                row.append("#")
            else:
                row.append("o")
        print(f"y={y:>2} | " + " ".join(row))
    print("       " + "-" * (2 * (max(xs) - min(xs) + 1)))
    print("  x =   " + " ".join(str(x % 10) for x in range(min(xs), max(xs) + 1)))

draw_points(points2)

y= 4 | o . . . .
y= 3 | . o . . o
y= 2 | . . o . .
y= 1 | o . . o .
       ----------
  x =   1 2 3 4 5


### Đếm tay example 2

Nhìn hình, thử "đặt thước kẻ" qua các điểm:

| Đường thẳng | Các điểm (index: tọa độ) | Số điểm |
|---|---|---:|
| đi xuống chéo, `x + y = 5` | 1: `(3,2)`, 3: `(4,1)`, 4: `(2,3)`, 5: `(1,4)` | **4** |
| đi lên thoai thoải (lên 1 khi sang 2) | 0: `(1,1)`, 1: `(3,2)`, 2: `(5,3)` | 3 |
| nằm ngang `y = 1` | 0: `(1,1)`, 3: `(4,1)` | 2 |
| thẳng đứng `x = 1` | 0: `(1,1)`, 5: `(1,4)` | 2 |

Vậy đáp án là **4**. Ta tô sáng hai đường đầu tiên để kiểm chứng.

In [3]:
print("Đường x + y = 5 (4 điểm):")
draw_points(points2, highlight={1, 3, 4, 5})
print()
print("Đường đi lên thoai thoải (3 điểm):")
draw_points(points2, highlight={0, 1, 2})

Đường x + y = 5 (4 điểm):
y= 4 | # . . . .
y= 3 | . # . . o
y= 2 | . . # . .
y= 1 | o . . # .
       ----------
  x =   1 2 3 4 5

Đường đi lên thoai thoải (3 điểm):
y= 4 | o . . . .
y= 3 | . o . . #
y= 2 | . . # . .
y= 1 | # . . o .
       ----------
  x =   1 2 3 4 5


### Hai quan sát nhỏ nhưng quan trọng

1. **Hai điểm phân biệt luôn xác định đúng một đường thẳng.** Vì vậy nếu `n = 1` thì đáp án là `1`, nếu `n = 2` thì đáp án là `2`. Tổng quát: đáp án luôn `>= min(n, 2)`.
2. Mọi đường thẳng "đáng quan tâm" (chứa ít nhất 2 điểm) đều **đi qua ít nhất một cặp điểm** của input. Ta không cần thử vô số đường thẳng trên mặt phẳng — chỉ cần thử các đường đi qua **cặp điểm** của input. Có `n·(n-1)/2` cặp như vậy.

Câu hỏi cốt lõi còn lại: **làm sao biết điểm thứ ba có nằm trên đường đi qua hai điểm đầu hay không?**

## 3. Kiến thức nền 1 — Độ dời `dx`, `dy` và hệ số góc (slope)

Đi từ điểm `A = (xA, yA)` tới điểm `B = (xB, yB)`:

- `dx = xB - xA` : sang phải bao nhiêu (âm = sang trái),
- `dy = yB - yA` : đi lên bao nhiêu (âm = đi xuống).

Cặp `(dx, dy)` gọi là **vector** (véc-tơ) từ A tới B — hiểu đơn giản là "mũi tên chỉ hướng và độ dài".

**Hệ số góc** (slope, độ dốc) của đường AB là `dy / dx` = "đi sang phải 1 bước thì lên bao nhiêu".

Ví dụ với example 2:

| A | B | dx | dy | slope = dy/dx |
|---|---|---:|---:|---:|
| `(1,1)` | `(3,2)` | 2 | 1 | 0.5 |
| `(1,1)` | `(5,3)` | 4 | 2 | 0.5 |
| `(3,2)` | `(4,1)` | 1 | -1 | -1.0 |
| `(1,1)` | `(4,1)` | 3 | 0 | 0.0 (nằm ngang) |

Hai cặp đầu có **cùng slope 0.5** và **cùng đi qua (1,1)** → cùng một đường thẳng. Đây chính là hạt giống của lời giải.

In [4]:
def delta(a, b):
    """Vector từ a tới b: (dx, dy)."""
    return b[0] - a[0], b[1] - a[1]

pairs = [((1, 1), (3, 2)), ((1, 1), (5, 3)), ((3, 2), (4, 1)), ((1, 1), (4, 1))]
for a, b in pairs:
    dx, dy = delta(a, b)
    print(f"A={a} -> B={b}: dx={dx:>2}, dy={dy:>2}, slope = {dy / dx}")

A=(1, 1) -> B=(3, 2): dx= 2, dy= 1, slope = 0.5
A=(1, 1) -> B=(5, 3): dx= 4, dy= 2, slope = 0.5
A=(3, 2) -> B=(4, 1): dx= 1, dy=-1, slope = -1.0
A=(1, 1) -> B=(4, 1): dx= 3, dy= 0, slope = 0.0


### Đường thẳng đứng: `dx = 0` → không chia được!

Với `A = (1,1)` và `B = (1,4)` (cùng `x = 1`): `dx = 0`, `dy = 3`. Slope `3 / 0` **không xác định** — trong Python, phép chia cho 0 ném lỗi `ZeroDivisionError`.

Ta dùng `try / except` để "bắt" lỗi mà không làm dừng chương trình:

```python
try:
    ... đoạn code có thể lỗi ...
except ZeroDivisionError:
    ... xử lý khi lỗi xảy ra ...
```

Đường thẳng đứng là **trường hợp đặc biệt số 1** mà mọi lời giải dùng phép chia phải tự xử lý.

In [5]:
a, b = (1, 1), (1, 4)
dx, dy = delta(a, b)
print("dx =", dx, ", dy =", dy)
try:
    print(dy / dx)
except ZeroDivisionError as e:
    print("Lỗi:", type(e).__name__, "-", e)

dx = 0 , dy = 3
Lỗi: ZeroDivisionError - division by zero


## 4. Kiến thức nền 2 — Tích chéo (cross product): kiểm tra thẳng hàng **không cần chia**

Ba điểm `A, B, C` **thẳng hàng** (collinear) khi C nằm trên đường AB, tức là hai vector `AB = (dx1, dy1)` và `AC = (dx2, dy2)` **cùng phương** (cùng hướng hoặc ngược hướng).

Nếu không có đường đứng, "cùng phương" nghĩa là cùng slope:

```text
dy1 / dx1 == dy2 / dx2
```

**Nhân chéo hai vế** (nhân cả hai vế với `dx1 · dx2`) để bỏ phép chia:

```text
dy1 · dx2 == dy2 · dx1      ⇔      dx1 · dy2 - dy1 · dx2 == 0
```

Biểu thức `dx1 · dy2 - dy1 · dx2` gọi là **tích chéo** (cross product 2D) của `AB` và `AC`:

```text
cross(A, B, C) = (xB - xA)·(yC - yA) - (yB - yA)·(xC - xA)
```

Kết luận: **`A, B, C` thẳng hàng ⇔ `cross(A, B, C) == 0`**.

Vì sao công thức này "xịn" hơn so sánh slope?

- Chỉ có **cộng, trừ, nhân số nguyên** → **không bao giờ** chia cho 0, **không bao giờ** sai số làm tròn.
- Đúng luôn cả với đường thẳng đứng (khi `dx1 = 0`), không cần `if` riêng.
- Với `|x|, |y| <= 10^4` thì `|dx|, |dy| <= 2·10^4`, tích chéo tối đa cỡ `8·10^8` — số nguyên Python không lo tràn số.

In [6]:
def cross(a, b, c):
    """Tích chéo của vector AB và AC. Bằng 0 <=> A, B, C thẳng hàng."""
    return (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])

A, B = (1, 1), (3, 2)
print("C=(5,3):", cross(A, B, (5, 3)), "-> thẳng hàng?", cross(A, B, (5, 3)) == 0)
print("C=(4,1):", cross(A, B, (4, 1)), "-> thẳng hàng?", cross(A, B, (4, 1)) == 0)
print("C=(-1,0):", cross(A, B, (-1, 0)), "-> thẳng hàng?", cross(A, B, (-1, 0)) == 0, "(nằm ngoài đoạn AB vẫn tính)")

# Đường thẳng đứng: không cần trường hợp đặc biệt
print("Đứng x=1, C=(1,7):", cross((1, 1), (1, 4), (1, 7)))
print("Đứng x=1, C=(2,7):", cross((1, 1), (1, 4), (2, 7)))

C=(5,3): 0 -> thẳng hàng? True
C=(4,1): -3 -> thẳng hàng? False
C=(-1,0): 0 -> thẳng hàng? True (nằm ngoài đoạn AB vẫn tính)
Đứng x=1, C=(1,7): 0
Đứng x=1, C=(2,7): -3


Tính tay một dòng để chắc chắn hiểu: `A=(1,1), B=(3,2), C=(4,1)`

```text
AB = (3-1, 2-1) = (2, 1)
AC = (4-1, 1-1) = (3, 0)
cross = 2·0 - 1·3 = -3  ≠ 0   → không thẳng hàng
```

**Bonus — dấu của tích chéo** (không cần cho bài này, nhưng rất hay gặp trong hình học tính toán): đi từ A sang B rồi nhìn C,

- `cross > 0`: C nằm **bên trái** (rẽ ngược chiều kim đồng hồ),
- `cross < 0`: C nằm **bên phải** (rẽ cùng chiều kim đồng hồ),
- `cross = 0`: C nằm **trên** đường AB.

In [7]:
def side(a, b, c):
    v = cross(a, b, c)
    return "trái" if v > 0 else ("phải" if v < 0 else "trên đường")

A, B = (0, 0), (4, 0)            # đi từ trái sang phải dọc trục X
for C in [(2, 3), (2, -3), (7, 0)]:
    print(f"C={C}: cross={cross(A, B, C):>3} -> C nằm {side(A, B, C)}")

C=(2, 3): cross= 12 -> C nằm trái
C=(2, -3): cross=-12 -> C nằm phải
C=(7, 0): cross=  0 -> C nằm trên đường


## 5. Cách 1 — Brute force: mọi cặp điểm + đếm bằng tích chéo

Từ quan sát ở mục 2: đường tốt nhất **đi qua một cặp điểm** nào đó. Vậy:

```text
nếu n <= 2: trả về n
best = 0
với mỗi cặp (i, j), i < j:          # cặp này định nghĩa một đường thẳng
    count = 0
    với mỗi k từ 0 tới n-1:
        nếu cross(P[i], P[j], P[k]) == 0:
            count += 1             # P[k] nằm trên đường (i, j)
    best = max(best, count)
trả về best
```

Chú ý: khi `k = i` hoặc `k = j` thì tích chéo tự động bằng 0 (vector `AA = (0,0)`), nên `count` **đã tính luôn** hai điểm `i` và `j` — không cần `+ 2` thủ công.

In [8]:
def max_points_bruteforce(points: List[List[int]]) -> int:
    n = len(points)
    if n <= 2:
        return n
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            count = 0
            for k in range(n):
                if cross(points[i], points[j], points[k]) == 0:
                    count += 1
            best = max(best, count)
    return best

print(max_points_bruteforce(points1), "(mong đợi 3)")
print(max_points_bruteforce(points2), "(mong đợi 4)")
print(max_points_bruteforce([[0, 0]]), "(một điểm -> 1)")
print(max_points_bruteforce([[0, 0], [5, -7]]), "(hai điểm -> 2)")

3 (mong đợi 3)
4 (mong đợi 4)
1 (một điểm -> 1)
2 (hai điểm -> 2)


### Trace cách 1 trên example 2

Với 6 điểm có `6·5/2 = 15` cặp. Bảng dưới liệt kê mỗi cặp `(i, j)` và các index `k` nằm trên đường đó:

| i | j | P[i] | P[j] | các k thẳng hàng | count |
|---:|---:|---|---|---|---:|
| 0 | 1 | (1,1) | (3,2) | 0, 1, 2 | 3 |
| 0 | 2 | (1,1) | (5,3) | 0, 1, 2 | 3 |
| 0 | 3 | (1,1) | (4,1) | 0, 3 | 2 |
| 0 | 4 | (1,1) | (2,3) | 0, 4 | 2 |
| 0 | 5 | (1,1) | (1,4) | 0, 5 | 2 |
| 1 | 2 | (3,2) | (5,3) | 0, 1, 2 | 3 |
| 1 | 3 | (3,2) | (4,1) | 1, 3, 4, 5 | **4** |
| 1 | 4 | (3,2) | (2,3) | 1, 3, 4, 5 | **4** |
| 1 | 5 | (3,2) | (1,4) | 1, 3, 4, 5 | **4** |
| 2 | 3 | (5,3) | (4,1) | 2, 3 | 2 |
| 2 | 4 | (5,3) | (2,3) | 2, 4 | 2 |
| 2 | 5 | (5,3) | (1,4) | 2, 5 | 2 |
| 3 | 4 | (4,1) | (2,3) | 1, 3, 4, 5 | **4** |
| 3 | 5 | (4,1) | (1,4) | 1, 3, 4, 5 | **4** |
| 4 | 5 | (2,3) | (1,4) | 1, 3, 4, 5 | **4** |

Để ý: đường `x + y = 5` (4 điểm) bị đếm lại **6 lần** (6 cặp trong 4 điểm), đường 3 điểm bị đếm lại 3 lần. Đây là sự lãng phí của brute force.

In [9]:
def trace_bruteforce(points):
    n = len(points)
    best = 0
    print(" i  j  P[i]     P[j]     | k thẳng hàng   | count | best")
    for i in range(n):
        for j in range(i + 1, n):
            on_line = [k for k in range(n) if cross(points[i], points[j], points[k]) == 0]
            best = max(best, len(on_line))
            print(f" {i}  {j}  {str(tuple(points[i])):<8} {str(tuple(points[j])):<8} | "
                  f"{str(on_line):<14} | {len(on_line):>5} | {best}")
    return best

print("Kết quả:", trace_bruteforce(points2))

 i  j  P[i]     P[j]     | k thẳng hàng   | count | best
 0  1  (1, 1)   (3, 2)   | [0, 1, 2]      |     3 | 3
 0  2  (1, 1)   (5, 3)   | [0, 1, 2]      |     3 | 3
 0  3  (1, 1)   (4, 1)   | [0, 3]         |     2 | 3
 0  4  (1, 1)   (2, 3)   | [0, 4]         |     2 | 3
 0  5  (1, 1)   (1, 4)   | [0, 5]         |     2 | 3
 1  2  (3, 2)   (5, 3)   | [0, 1, 2]      |     3 | 3
 1  3  (3, 2)   (4, 1)   | [1, 3, 4, 5]   |     4 | 4
 1  4  (3, 2)   (2, 3)   | [1, 3, 4, 5]   |     4 | 4
 1  5  (3, 2)   (1, 4)   | [1, 3, 4, 5]   |     4 | 4
 2  3  (5, 3)   (4, 1)   | [2, 3]         |     2 | 4
 2  4  (5, 3)   (2, 3)   | [2, 4]         |     2 | 4
 2  5  (5, 3)   (1, 4)   | [2, 5]         |     2 | 4
 3  4  (4, 1)   (2, 3)   | [1, 3, 4, 5]   |     4 | 4
 3  5  (4, 1)   (1, 4)   | [1, 3, 4, 5]   |     4 | 4
 4  5  (2, 3)   (1, 4)   | [1, 3, 4, 5]   |     4 | 4
Kết quả: 4


### Cách 1 vướng ở đâu?

Gọi `n = len(points)`.

- Số cặp `(i, j)`: `n·(n-1)/2`; mỗi cặp quét `n` điểm `k` → khoảng `n³/2` lần tính tích chéo.
- **Time:** `O(n³)`. **Space:** `O(1)`.

Với `n = 300`: khoảng `1.3·10⁷` lần gọi `cross` — với Python thuần (mỗi lần gọi hàm + 4 phép trừ + 2 phép nhân) thì đây là mức **khá sát giới hạn thời gian**, dễ bị TLE (Time Limit Exceeded). Quan trọng hơn: ta đang **đếm lại cùng một đường nhiều lần**. Cần một cách "gom" các điểm cùng đường lại trong **một lần quét**.

In [10]:
def count_cross_calls(n):
    return n * (n - 1) // 2 * n

for n in [6, 50, 300]:
    print(f"n = {n:>3}: số lần gọi cross ≈ {count_cross_calls(n):,}")

n =   6: số lần gọi cross ≈ 90
n =  50: số lần gọi cross ≈ 61,250
n = 300: số lần gọi cross ≈ 13,455,000


## 6. Insight: cố định một **điểm neo**, gom các điểm còn lại theo **hướng**

Hãy **cố định** một điểm `A` (gọi là **điểm neo** — anchor). Mọi đường thẳng đi qua `A` được xác định **duy nhất** bởi **hướng** của nó (slope). Vậy:

> Hai điểm `B` và `C` cùng nằm trên **một** đường thẳng đi qua `A` ⇔ slope `AB` == slope `AC`.

Vì cả hai đường cùng đi qua `A` **và** cùng độ dốc nên chúng trùng nhau — không thể là hai đường song song khác nhau.

Do đó, với điểm neo `A`, ta chỉ cần:

1. Tính slope từ `A` tới **mọi điểm khác**.
2. **Đếm** xem mỗi slope xuất hiện bao nhiêu lần (dùng `dict`).
3. Slope xuất hiện nhiều nhất `m` lần → đường đó có `m + 1` điểm (cộng thêm chính `A`).

Lặp với mọi điểm neo, lấy max. Mỗi điểm neo tốn `O(n)` → tổng `O(n²)`. Đây là đúng pattern "**cố định một phần tử + hash map**" giống Two Sum.

### Kiến thức nền 3 — `dict` để đếm tần suất

`dict` (dictionary, từ điển) lưu cặp **key → value**. Để đếm, key là "thứ cần đếm", value là "số lần gặp".

| Cách viết | Ý nghĩa |
|---|---|
| `counts = {}` | tạo dict rỗng |
| `counts.get(key, 0)` | lấy value của `key`; nếu chưa có thì trả `0` (không lỗi) |
| `counts[key] = counts.get(key, 0) + 1` | tăng bộ đếm của `key` thêm 1 |
| `max(counts.values(), default=0)` | giá trị đếm lớn nhất; nếu dict rỗng trả `0` thay vì báo lỗi |

`collections.Counter` là một dict chuyên đếm, viết gọn hơn: `c[key] += 1` mà không cần `get`.

In [11]:
from collections import Counter

slopes = [0.5, 0.5, 0.0, 2.0, 0.5]
counts = {}
for s in slopes:
    counts[s] = counts.get(s, 0) + 1
print("dict đếm :", counts)
print("max      :", max(counts.values(), default=0))

c = Counter()
for s in slopes:
    c[s] += 1
print("Counter  :", c, "-> phổ biến nhất:", c.most_common(1))

print("dict rỗng, có default:", max({}.values(), default=0))

dict đếm : {0.5: 3, 0.0: 1, 2.0: 1}
max      : 3
Counter  : Counter({0.5: 3, 0.0: 1, 2.0: 1}) -> phổ biến nhất: [(0.5, 3)]
dict rỗng, có default: 0


### Cách 2 — Điểm neo + slope dạng float

Bản đầu tiên ai cũng nghĩ ra: key là `dy / dx` (một số float). Đường thẳng đứng thì dùng key đặc biệt `float('inf')` (vô cực) để tránh chia cho 0.

```text
best = 1
với mỗi điểm neo i:
    counts = {}                       # PHẢI tạo mới cho mỗi điểm neo
    với mỗi j != i:
        dx, dy = P[j] - P[i]
        key = inf nếu dx == 0, ngược lại dy / dx
        counts[key] += 1
    best = max(best, max(counts) + 1)
```

In [12]:
def max_points_float(points: List[List[int]]) -> int:
    n = len(points)
    best = 1
    for i in range(n):
        counts = {}
        for j in range(n):
            if j == i:
                continue
            dx = points[j][0] - points[i][0]
            dy = points[j][1] - points[i][1]
            key = float("inf") if dx == 0 else dy / dx
            counts[key] = counts.get(key, 0) + 1
        best = max(best, max(counts.values(), default=0) + 1)
    return best

print(max_points_float(points1), "(mong đợi 3)")
print(max_points_float(points2), "(mong đợi 4)")
print(max_points_float([[0, 0], [0, 5], [0, -3]]), "(3 điểm trên đường đứng x = 0)")

3 (mong đợi 3)
4 (mong đợi 4)
3 (3 điểm trên đường đứng x = 0)


### Kiến thức nền 4 — Số thực `float` không chính xác tuyệt đối

Máy tính lưu `float` ở hệ nhị phân với khoảng **53 bit** độ chính xác (~15–16 chữ số thập phân). Nhiều số rất "đẹp" ở hệ 10 như `0.1` thì ở hệ 2 lại là số **vô hạn tuần hoàn**, nên bị **làm tròn** khi lưu.

Hệ quả: hai phép tính "về toán học bằng nhau" có thể cho hai float **khác nhau**, và — nguy hiểm hơn cho bài này — hai số **khác nhau** về toán học có thể bị làm tròn thành **cùng một** float.

In [13]:
print("0.1 + 0.2        =", 0.1 + 0.2)
print("0.1 + 0.2 == 0.3 :", 0.1 + 0.2 == 0.3)
print("1/3              =", 1 / 3)
print("Số chữ số in ra đầy đủ của 0.1:", f"{0.1:.25f}")

0.1 + 0.2        = 0.30000000000000004
0.1 + 0.2 == 0.3 : False
1/3              = 0.3333333333333333
Số chữ số in ra đầy đủ của 0.1: 0.1000000000000000055511151


### Ví dụ cụ thể: float làm cách 2 trả sai

Xét 3 điểm (một test nổi tiếng trong các thảo luận về bài này, từ phiên bản đề có tọa độ lớn hơn):

```text
O = (0, 0),  B = (94911151, 94911150),  C = (94911152, 94911151)
```

- slope `OB = 94911150 / 94911151`
- slope `OC = 94911151 / 94911152`

Hai phân số này **khác nhau** (hiệu của chúng là `1 / (94911151 · 94911152)` ≈ `1.1·10⁻¹⁶`), nhưng hiệu đó chỉ cỡ đúng **một bước nhảy** giữa hai float liên tiếp quanh số `0.99999...` (cũng ≈ `1.1·10⁻¹⁶`) → khi làm tròn, cả hai **rơi vào cùng một float**. Cách 2 sẽ tưởng `O, B, C` thẳng hàng và trả về `3`, trong khi đáp án đúng là `2`.

Tích chéo (chỉ dùng số nguyên) thì cho kết quả chính xác: `94911151·94911151 - 94911150·94911152 = 1 ≠ 0`.

In [14]:
O, B, C = (0, 0), (94911151, 94911150), (94911152, 94911151)
s1 = 94911150 / 94911151
s2 = 94911151 / 94911152
print("slope OB =", repr(s1))
print("slope OC =", repr(s2))
print("float coi là bằng nhau? ", s1 == s2)
print("tích chéo (số nguyên)  :", cross(O, B, C), "-> thẳng hàng?", cross(O, B, C) == 0)

bad = [list(O), list(B), list(C)]
print("max_points_float      :", max_points_float(bad), "<- SAI")
print("max_points_bruteforce :", max_points_bruteforce(bad), "<- đúng")

slope OB = 0.9999999894638303
slope OC = 0.9999999894638303
float coi là bằng nhau?  True
tích chéo (số nguyên)  : 1 -> thẳng hàng? False
max_points_float      : 3 <- SAI
max_points_bruteforce : 2 <- đúng


### Ghi chú trung thực: với constraints hiện tại thì sao?

Đề hiện hành có `|x|, |y| <= 10^4`, nên `|dx|, |dy| <= 2·10^4`. Hai phân số **khác nhau** `a/b` và `c/d` (với `b, d <= 2·10^4`) cách nhau ít nhất `1/(b·d) >= 2.5·10⁻⁹`, trong khi sai số làm tròn float của một slope cỡ tối đa `2·10^4` chỉ khoảng `10⁻¹²`. Vậy **trong phạm vi đề hiện tại**, `dy/dx` float tình cờ không va chạm, và cách 2 (có xử lý đường đứng) thường vẫn được Accepted.

Nhưng ta **không nên** dựa vào điều đó vì:

1. Phải **chứng minh** như trên mới dám tin; đổi constraints (tọa độ `10^9`) là sai ngay.
2. Các ngôn ngữ khác hoặc kiểu `float32` có độ chính xác thấp hơn nhiều.
3. Có chi tiết lắt léo như `0 / -3 = -0.0` (Python coi `-0.0 == 0.0` và cùng hash nên không sao, nhưng ở ngôn ngữ khác có thể thành hai key khác nhau).
4. Phỏng vấn viên thường **cố ý** hỏi câu này để xem bạn có biết tránh float không.

Ô code dưới kiểm chứng thực nghiệm trên phạm vi nhỏ `|dx|, |dy| <= 200`: số float khác nhau đúng bằng số phân số khác nhau (dùng `Fraction` — sẽ học ở cách 4 — làm "trọng tài" chính xác).

In [15]:
from fractions import Fraction

R = 200
float_keys, exact_keys = set(), set()
for dx in range(1, R + 1):
    for dy in range(-R, R + 1):
        float_keys.add(dy / dx)
        exact_keys.add(Fraction(dy, dx))
print("số slope khác nhau (float)   :", len(float_keys))
print("số slope khác nhau (chính xác):", len(exact_keys))
print("-0.0 == 0.0 ?", -0.0 == 0.0, "| 0 / -3 =", 0 / -3)

số slope khác nhau (float)   : 48927
số slope khác nhau (chính xác): 48927
-0.0 == 0.0 ? True | 0 / -3 = -0.0


## 7. Cách 3 (cách chính) — Slope dạng phân số tối giản `(dy, dx)` bằng số nguyên

Ý tưởng: **đừng chia**. Lưu slope dưới dạng **cặp số nguyên** `(dy, dx)`. Nhưng cùng một slope có thể viết nhiều kiểu:

| Viết | Giá trị | Vấn đề |
|---|---|---|
| `(1, 2)`, `(2, 4)`, `(3, 6)` | 1/2 | cần **rút gọn** (chia cho ước chung lớn nhất) |
| `(-1, 2)` và `(1, -2)` | -1/2 | cần **chuẩn hóa dấu** |
| `(3, 0)` và `(-5, 0)` | đường đứng | phải về **một** key duy nhất |
| `(0, 4)` và `(0, -1)` | nằm ngang | phải về **một** key duy nhất |

Ta cần một hàm `slope_key(dy, dx)` biến mọi cách viết của cùng một hướng thành **đúng một** tuple — gọi là **dạng chuẩn** (canonical form). Hai bước: **rút gọn bằng gcd** rồi **chuẩn hóa dấu**.

### Kiến thức nền 5 — `math.gcd` (ước chung lớn nhất)

`gcd(a, b)` (greatest common divisor) là số nguyên dương lớn nhất chia hết cả `a` và `b`. Chia cả tử và mẫu cho gcd → phân số **tối giản**: `gcd(4, 6) = 2` nên `4/6 → 2/3`.

Hành vi của `math.gcd` trong Python cần nhớ:

- Luôn trả về số **không âm**: `gcd(-12, 18) = 6`. Vì vậy chia cho gcd **không làm đổi dấu** của `dy`, `dx`.
- `gcd(0, b) = |b|`: `gcd(0, 7) = 7` → `(0, 7)` rút gọn thành `(0, 1)`; `(7, 0)` rút gọn thành `(1, 0)`. Hai trường hợp nằm ngang / thẳng đứng tự động gọn đẹp.
- `gcd(0, 0) = 0` → chia cho 0 sẽ lỗi. May mắn: các điểm **phân biệt** nên `(dx, dy)` không bao giờ là `(0, 0)`.

In [16]:
import math

print("gcd(4, 6)    =", math.gcd(4, 6))
print("gcd(-12, 18) =", math.gcd(-12, 18), "(luôn không âm)")
print("gcd(0, 7)    =", math.gcd(0, 7))
print("gcd(0, -7)   =", math.gcd(0, -7))
print("gcd(0, 0)    =", math.gcd(0, 0), "(chia cho số này sẽ lỗi!)")

dy, dx = -4, 6
g = math.gcd(dy, dx)
print(f"(-4, 6) chia gcd={g} -> ({dy // g}, {dx // g})")

gcd(4, 6)    = 2
gcd(-12, 18) = 6 (luôn không âm)
gcd(0, 7)    = 7
gcd(0, -7)   = 7
gcd(0, 0)    = 0 (chia cho số này sẽ lỗi!)
(-4, 6) chia gcd=2 -> (-2, 3)


### `gcd` chạy nhanh cỡ nào? — Thuật toán Euclid

`math.gcd` dùng **thuật toán Euclid**: `gcd(a, b) = gcd(b, a % b)`, dừng khi số thứ hai bằng 0. Mỗi 2 bước, số lớn **giảm ít nhất một nửa**, nên số bước chỉ cỡ `O(log C)` với `C` là độ lớn của số (ở đây `C <= 2·10^4` → chỉ vài chục bước). Tự cài để thấy từng bước:

In [17]:
def gcd_euclid_trace(a, b):
    a, b = abs(a), abs(b)
    while b != 0:
        print(f"gcd({a}, {b}) = gcd({b}, {a} % {b} = {a % b})")
        a, b = b, a % b
    print("=>", a)
    return a

gcd_euclid_trace(84, 36)

gcd(84, 36) = gcd(36, 84 % 36 = 12)
gcd(36, 12) = gcd(12, 36 % 12 = 0)
=> 12


12

### Quy tắc chuẩn hóa dấu

Sau khi chia gcd, `(dy, dx)` đã tối giản nhưng có thể còn **hai cách viết trái dấu** cho cùng một hướng: `(1, -2)` và `(-1, 2)`. (Hướng `A→B` và `B→A` là ngược nhau, nhưng **cùng một đường thẳng** — đường thẳng không có chiều.)

Quy ước: **làm cho `dx` dương**; nếu `dx = 0` (đường đứng) thì **làm cho `dy` dương**.

```text
nếu dx < 0 hoặc (dx == 0 và dy < 0):
    dy, dx = -dy, -dx
```

| `(dy, dx)` gốc | chia gcd | đổi dấu? | key |
|---|---|---|---|
| `(4, 6)` | `(2, 3)` | không | `(2, 3)` |
| `(-4, -6)` | `(-2, -3)` | có (`dx < 0`) | `(2, 3)` |
| `(4, -6)` | `(2, -3)` | có | `(-2, 3)` |
| `(0, -5)` | `(0, -1)` | có | `(0, 1)` ← mọi đường **ngang** |
| `(-7, 0)` | `(-1, 0)` | có (`dx = 0, dy < 0`) | `(1, 0)` ← mọi đường **đứng** |
| `(6, -6)` | `(1, -1)` | có | `(-1, 1)` |

Sau hai bước, **mỗi hướng có đúng một key**: hai `(dy, dx)` cho cùng slope ⇔ cùng key.

In [18]:
def slope_key(dy: int, dx: int) -> tuple:
    """Dạng chuẩn của hướng (dy, dx). Giả định (dy, dx) != (0, 0)."""
    g = math.gcd(dy, dx)
    dy //= g
    dx //= g
    if dx < 0 or (dx == 0 and dy < 0):
        dy, dx = -dy, -dx
    return (dy, dx)

for dy, dx in [(4, 6), (-4, -6), (4, -6), (0, -5), (-7, 0), (6, -6)]:
    print(f"slope_key({dy:>2}, {dx:>2}) = {slope_key(dy, dx)}")

slope_key( 4,  6) = (2, 3)
slope_key(-4, -6) = (2, 3)
slope_key( 4, -6) = (-2, 3)
slope_key( 0, -5) = (0, 1)
slope_key(-7,  0) = (1, 0)
slope_key( 6, -6) = (-1, 1)


### Kiến thức nền 6 — `tuple` làm key của dict (còn `list` thì không)

Key của dict phải **hashable** (băm được) — nói nôm na là **không thay đổi được** sau khi tạo, để dict tính "địa chỉ" một lần và tìm lại nhanh trung bình `O(1)`.

- `tuple` như `(2, 3)` không sửa được → làm key được.
- `list` như `[2, 3]` sửa được (`append`, gán phần tử) → **không** làm key được, Python báo `TypeError: unhashable type: 'list'`.

Đó là lý do `slope_key` trả về **tuple**.

In [19]:
counts = {}
for key in [(1, 2), (1, 2), (-1, 1), (1, 2)]:
    counts[key] = counts.get(key, 0) + 1
print("đếm theo tuple:", counts)

try:
    d = {[1, 2]: 1}
except TypeError as e:
    print("Dùng list làm key ->", type(e).__name__, ":", e)

đếm theo tuple: {(1, 2): 3, (-1, 1): 1}
Dùng list làm key -> TypeError : unhashable type: 'list'


### Ghép lại thành thuật toán — và một tối ưu nhỏ: chỉ xét `j > i`

Ở cách 2, với điểm neo `i` ta xét **mọi** `j != i`. Thật ra chỉ cần **`j > i`**:

> Gọi `L` là đường tốt nhất và `A` là điểm có **index nhỏ nhất** trên `L`. Khi `A` làm điểm neo, mọi điểm còn lại của `L` đều có index **lớn hơn** → đều được xét với `j > i` → `L` vẫn được đếm đủ.

Các điểm neo khác có thể "thấy thiếu" đường `L`, nhưng không sao: ta chỉ cần **một** lần nhìn thấy đầy đủ.

```text
nếu n <= 2: trả về n
best = 1
với mỗi i từ 0 tới n-1:
    counts = {}                                   # mới cho từng điểm neo
    với mỗi j từ i+1 tới n-1:
        key = slope_key(yj - yi, xj - xi)
        counts[key] += 1
    best = max(best, max(counts.values(), default=0) + 1)
trả về best
```

In [20]:
def max_points_gcd(points: List[List[int]]) -> int:
    n = len(points)
    if n <= 2:
        return n
    best = 1
    for i in range(n):
        xi, yi = points[i]
        counts = {}
        for j in range(i + 1, n):
            key = slope_key(points[j][1] - yi, points[j][0] - xi)
            counts[key] = counts.get(key, 0) + 1
        best = max(best, max(counts.values(), default=0) + 1)
    return best

print(max_points_gcd(points1), "(mong đợi 3)")
print(max_points_gcd(points2), "(mong đợi 4)")
print(max_points_gcd([[0, 0], [94911151, 94911150], [94911152, 94911151]]), "(bẫy float, mong đợi 2)")

3 (mong đợi 3)
4 (mong đợi 4)
2 (bẫy float, mong đợi 2)


### Trace cách 3 trên example 2 `[[1,1],[3,2],[5,3],[4,1],[2,3],[1,4]]`

| neo i | j | P[j] | dy | dx | key | counts sau bước |
|---:|---:|---|---:|---:|---|---|
| 0 `(1,1)` | 1 | (3,2) | 1 | 2 | `(1,2)` | `{(1,2):1}` |
| | 2 | (5,3) | 2 | 4 | `(1,2)` | `{(1,2):2}` |
| | 3 | (4,1) | 0 | 3 | `(0,1)` | `+ (0,1):1` |
| | 4 | (2,3) | 2 | 1 | `(2,1)` | `+ (2,1):1` |
| | 5 | (1,4) | 3 | 0 | `(1,0)` | `+ (1,0):1` → max 2 → **3 điểm** |
| 1 `(3,2)` | 2 | (5,3) | 1 | 2 | `(1,2)` | `{(1,2):1}` |
| | 3 | (4,1) | -1 | 1 | `(-1,1)` | `(-1,1):1` |
| | 4 | (2,3) | 1 | -1 | `(-1,1)` ← đổi dấu | `(-1,1):2` |
| | 5 | (1,4) | 2 | -2 | `(-1,1)` ← chia gcd 2, đổi dấu | `(-1,1):3` → max 3 → **4 điểm** |
| 2 `(5,3)` | 3, 4, 5 | | | | `(2,1)`, `(0,1)`, `(-1,4)` | mỗi key 1 → 2 điểm |
| 3 `(4,1)` | 4, 5 | | | | `(-1,1)`, `(-1,1)` | max 2 → 3 điểm (thấy thiếu điểm 1 — không sao!) |
| 4 `(2,3)` | 5 | | | | `(-1,1)` | 2 điểm |
| 5 `(1,4)` | — | | | | | dict rỗng → 1 điểm |

Điểm neo 1 là điểm có index nhỏ nhất trên đường `x + y = 5`, nên nó thấy đủ cả 3 điểm còn lại. Ba cặp `(-1,1)`, `(1,-1)`, `(2,-2)` gộp về **cùng key** `(-1,1)` nhờ chia gcd + chuẩn hóa dấu — đây là chỗ cách làm "ngây thơ" hay sai.

In [21]:
def trace_max_points_gcd(points):
    n = len(points)
    best = 1
    for i in range(n):
        xi, yi = points[i]
        counts = {}
        for j in range(i + 1, n):
            dy, dx = points[j][1] - yi, points[j][0] - xi
            key = slope_key(dy, dx)
            counts[key] = counts.get(key, 0) + 1
            print(f"   neo {i} -> j={j} {tuple(points[j])}: (dy,dx)=({dy:>2},{dx:>2}) key={key}")
        top = max(counts.values(), default=0)
        best = max(best, top + 1)
        print(f"neo {i} {tuple(points[i])}: counts={counts} -> {top + 1} điểm, best={best}")
    return best

print("Kết quả:", trace_max_points_gcd(points2))

   neo 0 -> j=1 (3, 2): (dy,dx)=( 1, 2) key=(1, 2)
   neo 0 -> j=2 (5, 3): (dy,dx)=( 2, 4) key=(1, 2)
   neo 0 -> j=3 (4, 1): (dy,dx)=( 0, 3) key=(0, 1)
   neo 0 -> j=4 (2, 3): (dy,dx)=( 2, 1) key=(2, 1)
   neo 0 -> j=5 (1, 4): (dy,dx)=( 3, 0) key=(1, 0)
neo 0 (1, 1): counts={(1, 2): 2, (0, 1): 1, (2, 1): 1, (1, 0): 1} -> 3 điểm, best=3
   neo 1 -> j=2 (5, 3): (dy,dx)=( 1, 2) key=(1, 2)
   neo 1 -> j=3 (4, 1): (dy,dx)=(-1, 1) key=(-1, 1)
   neo 1 -> j=4 (2, 3): (dy,dx)=( 1,-1) key=(-1, 1)
   neo 1 -> j=5 (1, 4): (dy,dx)=( 2,-2) key=(-1, 1)
neo 1 (3, 2): counts={(1, 2): 1, (-1, 1): 3} -> 4 điểm, best=4
   neo 2 -> j=3 (4, 1): (dy,dx)=(-2,-1) key=(2, 1)
   neo 2 -> j=4 (2, 3): (dy,dx)=( 0,-3) key=(0, 1)
   neo 2 -> j=5 (1, 4): (dy,dx)=( 1,-4) key=(-1, 4)
neo 2 (5, 3): counts={(2, 1): 1, (0, 1): 1, (-1, 4): 1} -> 2 điểm, best=4
   neo 3 -> j=4 (2, 3): (dy,dx)=( 2,-2) key=(-1, 1)
   neo 3 -> j=5 (1, 4): (dy,dx)=( 3,-3) key=(-1, 1)
neo 3 (4, 1): counts={(-1, 1): 2} -> 3 điểm, best=4
   neo 

### Độ phức tạp cách 3

Gọi `n = len(points)`, `C` = độ lớn tối đa của `|dx|, |dy|` (ở đây `C <= 2·10^4`).

- **Time:** `O(n² · log C)` — có `n·(n-1)/2` cặp `(i, j)`, mỗi cặp một lần `gcd` tốn `O(log C)` và một thao tác dict trung bình `O(1)`. Với `n = 300` chỉ khoảng `44 850` lần gọi `slope_key`, so với `13 455 000` lần gọi `cross` ở brute force (ít hơn **300 lần**).
- **Space:** `O(n)` — dict `counts` của một điểm neo có tối đa `n - 1` key, và được tạo mới cho mỗi điểm neo.

**Tối ưu dừng sớm (tùy chọn):** với `j > i`, điểm neo `i` thấy tối đa `n - i` điểm (tính cả chính nó). Nếu `best >= n - i` thì các điểm neo còn lại không thể vượt `best` → `break`. Không đổi Big-O nhưng giúp chạy nhanh hơn khi có đường rất dài.

## 8. Code nộp LeetCode

Bản submit dùng cách 3, viết gọn `slope_key` ngay trong vòng lặp, có thêm dừng sớm. Đúng signature `maxPoints(self, points: List[List[int]]) -> int`, không `print`.

In [22]:
from math import gcd


class Solution:
    def maxPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        if n <= 2:
            return n
        best = 1
        for i in range(n):
            if best >= n - i:              # dừng sớm: không thể vượt best nữa
                break
            xi, yi = points[i]
            counts = {}
            for j in range(i + 1, n):
                dy = points[j][1] - yi
                dx = points[j][0] - xi
                g = gcd(dy, dx)            # điểm phân biệt nên g > 0
                dy //= g
                dx //= g
                if dx < 0 or (dx == 0 and dy < 0):
                    dy, dx = -dy, -dx
                key = (dy, dx)
                counts[key] = counts.get(key, 0) + 1
            best = max(best, max(counts.values()) + 1)
        return best


sol = Solution()
tests = [
    (points1, 3),                                              # example 1
    (points2, 4),                                              # example 2
    ([[0, 0]], 1),                                             # n = 1
    ([[0, 0], [-3, 7]], 2),                                    # n = 2
    ([[0, 0], [0, 5], [0, -9], [1, 1]], 3),                    # đường thẳng đứng
    ([[-4, 2], [0, 2], [7, 2], [1, 3]], 3),                    # đường nằm ngang
    ([[0, 0], [1, 1], [-1, -1], [2, 3]], 3),                   # điểm neo ở giữa (cần đổi dấu)
    ([[0, 0], [1, 1], [0, 1], [1, 2]], 2),                     # hai đường song song
    ([[0, 0], [94911151, 94911150], [94911152, 94911151]], 2), # bẫy float
    ([[-10000, -10000], [0, 0], [10000, 10000], [10000, -10000]], 3),  # tọa độ biên
]
for pts, expected in tests:
    got = sol.maxPoints(pts)
    assert got == expected, (pts, got, expected)
print("Tất cả", len(tests), "test PASS!")

Tất cả 10 test PASS!


### Vì sao trong `Solution` dùng `max(counts.values())` không cần `default`?

Nhờ dòng dừng sớm: điểm neo `i` chỉ được xử lý khi `best < n - i`. Vì `best >= 1` nên `n - i >= 2`, tức là có ít nhất một `j > i` → `counts` không bao giờ rỗng. Nếu bỏ dòng dừng sớm thì điểm neo cuối có `counts` rỗng và `max(...)` sẽ lỗi `ValueError` — xem lỗi 4 ở mục 13. Khi không chắc, cứ viết `max(..., default=0)` cho an toàn.

### Kiểm thử đối chiếu chéo

Brute force chậm nhưng **dễ tin là đúng** vì chỉ dùng tích chéo số nguyên. Ta sinh ngẫu nhiên nhiều bộ điểm nhỏ trên lưới `[-3, 3] × [-3, 3]` (lưới nhỏ để có **nhiều** điểm thẳng hàng), dùng `random.sample` để các điểm **phân biệt** đúng như đề, cố định `random.seed` để lần nào chạy cũng cùng bộ test.

In [23]:
import random

random.seed(149)
grid = [[x, y] for x in range(-3, 4) for y in range(-3, 4)]
for _ in range(1500):
    n = random.randint(1, 12)
    pts = random.sample(grid, n)                 # n điểm phân biệt
    expected = max_points_bruteforce(pts)
    assert max_points_float(pts) == expected, pts
    assert max_points_gcd(pts) == expected, pts
    assert sol.maxPoints(pts) == expected, pts
print("1500 bộ test ngẫu nhiên: brute force, float, gcd, Solution đều khớp!")

# Trường hợp lớn nhất: 300 điểm cùng nằm trên đường y = 3x - 7
big = [[x, 3 * x - 7] for x in range(-150, 150)]
random.shuffle(big)
print("300 điểm thẳng hàng ->", sol.maxPoints(big))

1500 bộ test ngẫu nhiên: brute force, float, gcd, Solution đều khớp!
300 điểm thẳng hàng -> 300


## 9. Cách 4 — Dùng `fractions.Fraction` làm key

Thư viện chuẩn có sẵn kiểu **phân số chính xác** `Fraction(tử, mẫu)`: nó **tự rút gọn** bằng gcd và **tự chuẩn hóa dấu** (mẫu luôn dương). Vậy thay `dy / dx` (float) bằng `Fraction(dy, dx)` là có ngay key chính xác — chỉ cần xử lý riêng đường đứng (`dx = 0`, vì `Fraction(3, 0)` cũng lỗi chia cho 0).

In [24]:
print("Fraction(2, 4)  =", Fraction(2, 4), "| bằng Fraction(1, 2)?", Fraction(2, 4) == Fraction(1, 2))
print("Fraction(1, -2) =", Fraction(1, -2), "| bằng Fraction(-1, 2)?", Fraction(1, -2) == Fraction(-1, 2))
print("Fraction(0, -5) =", Fraction(0, -5))
print("hash bằng nhau? ", hash(Fraction(3, 6)) == hash(Fraction(1, 2)))
print("Fraction coi 2 slope của bẫy float là bằng nhau?", Fraction(94911150, 94911151) == Fraction(94911151, 94911152))
try:
    Fraction(3, 0)
except ZeroDivisionError as e:
    print("Fraction(3, 0) ->", type(e).__name__)

Fraction(2, 4)  = 1/2 | bằng Fraction(1, 2)? True
Fraction(1, -2) = -1/2 | bằng Fraction(-1, 2)? True
Fraction(0, -5) = 0
hash bằng nhau?  True
Fraction coi 2 slope của bẫy float là bằng nhau? False
Fraction(3, 0) -> ZeroDivisionError


In [25]:
def max_points_fraction(points: List[List[int]]) -> int:
    n = len(points)
    if n <= 2:
        return n
    best = 1
    for i in range(n):
        counts = {}
        for j in range(i + 1, n):
            dx = points[j][0] - points[i][0]
            dy = points[j][1] - points[i][1]
            key = "vertical" if dx == 0 else Fraction(dy, dx)
            counts[key] = counts.get(key, 0) + 1
        best = max(best, max(counts.values(), default=0) + 1)
    return best

print(max_points_fraction(points1), max_points_fraction(points2),
      max_points_fraction([[0, 0], [94911151, 94911150], [94911152, 94911151]]))

3 4 2


**Nhận xét cách 4:** cùng `O(n² log C)` như cách 3, code ngắn và khó sai hơn. Nhược điểm: tạo object `Fraction` **chậm hơn** tạo tuple số nguyên (đo thử thường chậm khoảng 2 lần trở lên, vì module `fractions` viết bằng Python thuần), và trong phỏng vấn người ta thường muốn thấy bạn **tự** chuẩn hóa `(dy, dx)`. Dùng khi cần code nhanh, đúng; hiểu cách 3 để biết `Fraction` làm gì bên trong.

## 10. Cách 5 — Biểu diễn cả đường thẳng bằng `ax + by = c`

Cách 3, 4 chỉ phân biệt **hướng** — nên mới cần cố định điểm neo (để hai đường song song không bị gộp). Nếu key mô tả **cả đường thẳng** (hướng **và** vị trí) thì có thể dùng **một dict chung** cho mọi cặp điểm.

Đường qua `P1 = (x1, y1)` và `P2 = (x2, y2)`:

```text
a = y2 - y1,   b = x1 - x2,   c = a·x1 + b·y1         (đường:  a·x + b·y = c)
```

Kiểm tra: thế `P2` vào: `a·x2 + b·y2 - c = (y2-y1)(x2-x1) + (x1-x2)(y2-y1) = 0` ✓.

Chuẩn hóa giống cách 3: chia `(a, b, c)` cho `gcd(a, b, c)`, rồi đổi dấu để `a > 0` (hoặc `a = 0` và `b > 0`).

Ví dụ: đường qua `(1,4)` và `(4,1)`: `a = -3, b = -3, c = -3·1 + (-3)·4 = -15` → chia 3: `(-1, -1, -5)` → đổi dấu: `(1, 1, 5)`, tức **`x + y = 5`** — đúng đường 4 điểm ta đã thấy.

In [26]:
def line_key(p1, p2):
    """Dạng chuẩn (a, b, c) của đường a*x + b*y = c qua p1, p2 (phân biệt)."""
    (x1, y1), (x2, y2) = p1, p2
    a = y2 - y1
    b = x1 - x2
    c = a * x1 + b * y1
    g = math.gcd(math.gcd(a, b), c)        # gcd(a, b) > 0 vì p1 != p2
    a, b, c = a // g, b // g, c // g
    if a < 0 or (a == 0 and b < 0):
        a, b, c = -a, -b, -c
    return (a, b, c)

print("qua (1,4),(4,1):", line_key((1, 4), (4, 1)), "-> x + y = 5")
print("qua (2,3),(3,2):", line_key((2, 3), (3, 2)), "-> cùng đường")
print("qua (0,1),(1,2):", line_key((0, 1), (1, 2)), "-> y = x + 1")
print("qua (0,0),(1,1):", line_key((0, 0), (1, 1)), "-> y = x (song song, khác key)")
print("qua (1,1),(1,4):", line_key((1, 1), (1, 4)), "-> x = 1 (đường đứng)")

qua (1,4),(4,1): (1, 1, 5) -> x + y = 5
qua (2,3),(3,2): (1, 1, 5) -> cùng đường
qua (0,1),(1,2): (1, -1, -1) -> y = x + 1
qua (0,0),(1,1): (1, -1, 0) -> y = x (song song, khác key)
qua (1,1),(1,4): (1, 0, 1) -> x = 1 (đường đứng)


In [27]:
from collections import defaultdict

def max_points_lines(points: List[List[int]]) -> int:
    n = len(points)
    if n <= 2:
        return n
    members = defaultdict(set)              # key đường -> tập index điểm trên đường
    for i in range(n):
        for j in range(i + 1, n):
            members[line_key(points[i], points[j])].update((i, j))
    return max(len(s) for s in members.values())

print(max_points_lines(points1), max_points_lines(points2))

# Bonus của cách 5: liệt kê được MỌI đường có >= 3 điểm
members = defaultdict(set)
for i in range(len(points2)):
    for j in range(i + 1, len(points2)):
        members[line_key(points2[i], points2[j])].update((i, j))
for key, s in members.items():
    if len(s) >= 3:
        a, b, c = key
        print(f"đường {a}·x + ({b})·y = {c}  chứa các index {sorted(s)}")

random.seed(7)
for _ in range(500):
    pts = random.sample(grid, random.randint(1, 12))
    assert max_points_lines(pts) == max_points_fraction(pts) == max_points_bruteforce(pts), pts
print("Cách 4, 5 khớp brute force trên 500 bộ ngẫu nhiên!")

3 4
đường 1·x + (-2)·y = -1  chứa các index [0, 1, 2]
đường 1·x + (1)·y = 5  chứa các index [1, 3, 4, 5]


Cách 4, 5 khớp brute force trên 500 bộ ngẫu nhiên!


**`defaultdict(set)`** là dict mà khi truy cập key chưa có, nó **tự tạo** giá trị mặc định `set()` (tập rỗng) — không cần `if key not in d`. `set.update((i, j))` thêm cả `i` và `j` vào tập (trùng thì bỏ qua).

**Nhận xét cách 5:** Time `O(n² log C)`, nhưng **Space `O(n²)`** (tệ nhất mỗi cặp là một đường riêng → `n(n-1)/2` key). Không cần điểm neo và có thể trả lời thêm các câu hỏi như "có bao nhiêu đường chứa ≥ 3 điểm", "liệt kê các điểm trên đường tốt nhất".

## 11. Vì sao cách 3 đúng? (invariant bằng lời)

**Bổ đề 1 — key là dạng chuẩn.** Với hai vector khác `(0, 0)` là `(dy1, dx1)` và `(dy2, dx2)`: chúng cùng phương ⇔ `slope_key` của chúng bằng nhau.
- (⇐) key bằng nhau nghĩa là cả hai đều là bội số (khác 0) của cùng một vector tối giản → cùng phương.
- (⇒) cùng phương thì mỗi vector bằng `±k ×` một vector tối giản duy nhất (bỏ qua dấu); chia gcd lấy đúng vector tối giản đó (có thể trái dấu), và quy tắc "`dx > 0`, hoặc `dx = 0` thì `dy > 0`" chọn ra **đúng một** trong hai dấu → key giống nhau.

**Bổ đề 2 — cố định điểm neo thì hướng ⇔ đường.** Với điểm neo `A`, hai điểm `B, C` cùng nằm trên một đường qua `A` ⇔ `AB`, `AC` cùng phương ⇔ (bổ đề 1) cùng key. Vậy với điểm neo `i`, `counts[key] + 1` **chính xác** bằng số điểm (có index `>= i`) trên đường qua `P[i]` theo hướng `key`.

**Không bỏ sót.** Gọi `L` là đường có nhiều điểm nhất (`m` điểm) và `A` là điểm của `L` có index nhỏ nhất. Khi `A` là điểm neo, `m - 1` điểm còn lại của `L` đều có index lớn hơn → đều rơi vào cùng một key → `best >= m`.

**Không đếm thừa.** Mọi giá trị `counts[key] + 1` đều là số điểm thật sự nằm trên **một** đường thẳng (bổ đề 2), nên `best <= m`. Vậy `best = m`.

**Dừng sớm vẫn đúng.** Điểm neo `i` chỉ thấy các điểm index `>= i`, tối đa `n - i` điểm; khi `best >= n - i` thì mọi điểm neo `>= i` không thể cho kết quả lớn hơn.

## 12. So sánh các cách

`n = len(points)`, `C` = độ lớn tối đa của `|dx|, |dy|` (ở đây `2·10^4`).

| Cách | Ý tưởng | Time | Space | Chính xác? | Nhận xét |
|---|---|---:|---:|---|---|
| 1. Brute force + tích chéo | Mọi cặp `(i, j)`, quét mọi `k` | `O(n³)` | `O(1)` | Tuyệt đối (số nguyên) | Dễ tin đúng; dùng làm "trọng tài" để test; sát giới hạn TLE |
| 2. Neo + slope float | Dict đếm `dy/dx` | `O(n²)` | `O(n)` | **Không** đảm bảo | Phải xử lý `dx = 0`; sai với tọa độ lớn |
| 3. Neo + `(dy, dx)` tối giản | `gcd` + chuẩn hóa dấu | `O(n² log C)` | `O(n)` | Tuyệt đối | **Cách chính**, chuẩn phỏng vấn |
| 4. Neo + `Fraction` | Phân số chính xác có sẵn | `O(n² log C)` | `O(n)` | Tuyệt đối | Ngắn gọn nhưng chậm hơn cách 3 (tạo object Fraction) |
| 5. Dict chung theo `(a, b, c)` | Key là cả đường thẳng | `O(n² log C)` | `O(n²)` | Tuyệt đối | Không cần neo; liệt kê được mọi đường |

## 13. Lỗi người mới hay gặp

### Lỗi 1: Chia `dy / dx` mà quên đường thẳng đứng
`dx = 0` → `ZeroDivisionError`. Sửa: key riêng cho đường đứng, hoặc tốt hơn là dùng `(dy, dx)` số nguyên (cách 3 tự xử lý được).

### Lỗi 2: Tin tuyệt đối vào float
Xem ví dụ `94911150/94911151 == 94911151/94911152` ở mục 6. Sửa: dùng số nguyên (tích chéo, `(dy, dx)` tối giản, `Fraction`).

### Lỗi 3: Rút gọn bằng gcd nhưng **quên chuẩn hóa dấu**
`(1, 1)` và `(-1, -1)` là cùng một đường nhưng thành hai key → khi điểm neo nằm **giữa** hai điểm khác sẽ đếm thiếu.

### Lỗi 4: `max()` trên dict rỗng
Điểm neo cuối cùng (với `j > i`) không có `j` nào → `max(counts.values())` báo `ValueError`. Sửa: `max(..., default=0)` hoặc dừng sớm như bản `Solution`.

### Lỗi 5: Dùng **một dict chung** cho mọi điểm neo (không tạo lại `counts`)
Key `(dy, dx)` chỉ mô tả **hướng**, không mô tả vị trí → các đường **song song** bị gộp lại. Thậm chí có thể trả về số lớn hơn cả số điểm! Sửa: `counts = {}` **bên trong** vòng `for i` (hoặc dùng key cả đường `(a, b, c)` như cách 5).

### Lỗi 6: Quên `+ 1` cho chính điểm neo
`counts[key]` chỉ đếm các điểm **khác** điểm neo. Đường có `counts[key] = 3` chứa `4` điểm.

In [28]:
def wrong_no_sign(points):                      # Lỗi 3: chỉ chia gcd, không đổi dấu
    n = len(points)
    if n <= 2:
        return n
    best = 1
    for i in range(n):
        counts = {}
        for j in range(i + 1, n):
            dy, dx = points[j][1] - points[i][1], points[j][0] - points[i][0]
            g = math.gcd(dy, dx)
            key = (dy // g, dx // g)
            counts[key] = counts.get(key, 0) + 1
        best = max(best, max(counts.values(), default=0) + 1)
    return best

def wrong_shared_dict(points):                  # Lỗi 5: counts tạo một lần duy nhất
    n = len(points)
    if n <= 2:
        return n
    counts, best = {}, 1
    for i in range(n):
        for j in range(i + 1, n):
            key = slope_key(points[j][1] - points[i][1], points[j][0] - points[i][0])
            counts[key] = counts.get(key, 0) + 1
            best = max(best, counts[key] + 1)
    return best

def wrong_empty_max(points):                    # Lỗi 4: max trên dict rỗng
    best = 1
    for i in range(len(points)):
        counts = {}
        for j in range(i + 1, len(points)):
            key = slope_key(points[j][1] - points[i][1], points[j][0] - points[i][0])
            counts[key] = counts.get(key, 0) + 1
        best = max(best, max(counts.values()) + 1)
    return best

mid = [[0, 0], [1, 1], [-1, -1]]
print("Lỗi 3 trên", mid, ":", wrong_no_sign(mid), "| đúng:", max_points_gcd(mid))
par = [[0, 0], [1, 1], [0, 1], [1, 2]]
print("Lỗi 5 trên hai đường song song:", wrong_shared_dict(par), "| đúng:", max_points_gcd(par))
print("Lỗi 5 trên example 2 (6 điểm!):", wrong_shared_dict(points2), "| đúng:", max_points_gcd(points2))
try:
    wrong_empty_max(points2)
except ValueError as e:
    print("Lỗi 4 ->", type(e).__name__, ":", e)

Lỗi 3 trên [[0, 0], [1, 1], [-1, -1]] : 2 | đúng: 3
Lỗi 5 trên hai đường song song: 3 | đúng: 2
Lỗi 5 trên example 2 (6 điểm!): 7 | đúng: 4
Lỗi 4 -> ValueError : max() iterable argument is empty


Đọc kết quả:

- **Lỗi 3**: điểm neo `(0,0)` thấy `(1,1)` với key `(1,1)` và `(-1,-1)` với key `(-1,-1)` → hai key khác nhau, đếm được 2 thay vì 3.
- **Lỗi 5** có hai tác hại:
  - Hai đường song song `y = x` và `y = x + 1` đều có hướng `(1, 1)` → cặp `(0,1)` và cặp `(2,3)` bị cộng chung → ra 3 thay vì 2.
  - Trên example 2 trả về **7** — nhiều hơn cả 6 điểm của input! Vì dict chung đếm **cặp điểm** chứ không phải điểm: đường `x + y = 5` có 4 điểm tạo ra **6 cặp** (`1-3, 1-4, 1-5, 3-4, 3-5, 4-5`), cả 6 cặp đều có key `(-1, 1)` → bộ đếm lên 6, cộng 1 thành 7. Dict riêng cho từng điểm neo tránh được cả hai vấn đề, vì khi đó mỗi `j` chỉ được đếm **một lần** so với **một** điểm cố định.
- **Lỗi 4**: điểm neo cuối (index 5) không có `j > 5` nào → `max()` của dãy rỗng.

## 14. Mẹo nhận dạng pattern

- Đề hỏi về **thẳng hàng / cùng đường / cùng hướng** với tọa độ nguyên → nghĩ ngay tới **tích chéo** hoặc **phân số tối giản**, **tránh float**.
- "Đếm nhóm lớn nhất có chung một tính chất **so với một phần tử**" → **cố định một phần tử (điểm neo) + hash map đếm** — cùng họ với Two Sum, Number of Boomerangs.
- Muốn dùng một đại lượng "phân số" làm key của dict → **dạng chuẩn**: rút gọn bằng `gcd` + **quy ước dấu** để mỗi giá trị chỉ có một cách viết.
- `n <= 300` (hoặc vài nghìn) gợi ý lời giải `O(n²)` là đủ; `O(n³)` thì sát giới hạn.
- Brute force số nguyên luôn là **trọng tài** tốt để kiểm thử lời giải nhanh trên input ngẫu nhiên nhỏ.

## 15. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 1232 — Check If It Is a Straight Line (Easy) | Kiểm tra **tất cả** điểm thẳng hàng → tích chéo với 2 điểm đầu (bài tập 5) |
| LeetCode 2280 — Minimum Lines to Represent a Line Chart (Medium) | So sánh slope các đoạn liên tiếp; tọa độ tới `10^9` nên **float sai thật** → phải nhân chéo |
| LeetCode 447 — Number of Boomerangs (Medium) | Cùng pattern **điểm neo + dict đếm**, nhưng key là **khoảng cách bình phương** |
| LeetCode 356 — Line Reflection (Medium, Premium) | Hình học trên điểm nguyên + set/dict |
| Phiên bản cũ của 149 có **điểm trùng** | Đếm riêng số điểm trùng với điểm neo rồi cộng vào mọi đường (bài tập 6) |

## 16. Bài tập vận dụng

Làm theo thứ tự từ dễ tới khó. Gợi ý ở mục 17, đáp án ở **cuối notebook**.

### Bài 1 — Tính tay tích chéo (không chạy code)

Với mỗi bộ ba điểm, tính `AB`, `AC`, `cross(A, B, C)` và kết luận có thẳng hàng không:

- (a) `A=(0,0), B=(2,4), C=(3,6)`
- (b) `A=(1,2), B=(4,3), C=(7,5)`
- (c) `A=(2,-1), B=(2,5), C=(2,100)`
- (d) `A=(-1,-1), B=(1,0), C=(5,2)`

### Bài 2 — Đoán output

Không chạy code, hãy đoán kết quả của từng dòng:

```python
math.gcd(-12, 18)
math.gcd(0, 0)
slope_key(-6, -9)
slope_key(5, -10)
slope_key(0, -8)
slope_key(-3, 0)
```

### Bài 3 — Điền TODO: hàm chuẩn hóa hướng

Hoàn thiện `exercise_slope_key(dy, dx)` trả về tuple dạng chuẩn như `slope_key` (rút gọn bằng gcd, `dx > 0`, hoặc `dx = 0` thì `dy > 0`).

In [29]:
def exercise_slope_key(dy: int, dx: int) -> tuple:
    # TODO 1: g = gcd(dy, dx), chia cả dy và dx cho g (dùng //)
    # TODO 2: nếu dx < 0, hoặc dx == 0 và dy < 0 thì đổi dấu cả hai
    # TODO 3: trả về (dy, dx)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_slope_key(4, 6) == (2, 3)
# assert exercise_slope_key(-4, -6) == (2, 3)
# assert exercise_slope_key(4, -6) == (-2, 3)
# assert exercise_slope_key(0, -5) == (0, 1)
# assert exercise_slope_key(-7, 0) == (1, 0)

### Bài 4 — Tự viết lại cách 3

Không nhìn code phía trên, viết `exercise_max_points(points)` theo ý tưởng **điểm neo + dict đếm key `(dy, dx)` tối giản**. Nhớ: `n <= 2`, tạo dict mới cho mỗi điểm neo, `+ 1` cho chính điểm neo, dict có thể rỗng.

In [30]:
def exercise_max_points(points: List[List[int]]) -> int:
    # TODO: điểm neo i, j > i, key chuẩn hóa, đếm bằng dict, cập nhật best
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_max_points([[1,1],[2,2],[3,3]]) == 3
# assert exercise_max_points([[1,1],[3,2],[5,3],[4,1],[2,3],[1,4]]) == 4
# assert exercise_max_points([[0,0]]) == 1
# assert exercise_max_points([[0,0],[1,1],[-1,-1],[5,0]]) == 3

### Bài 5 — Biến thể: LeetCode 1232 — Check If It Is a Straight Line

Cho `coordinates` (`2 <= len <= 1000`, `-10^4 <= tọa độ <= 10^4`, không có điểm trùng). Trả về `True` nếu **tất cả** các điểm nằm trên cùng một đường thẳng.

```python
check_straight_line([[1,2],[2,3],[3,4],[4,5],[5,6],[6,7]])  -> True
check_straight_line([[1,1],[2,2],[3,4],[4,5],[5,6],[7,7]])  -> False
```

Yêu cầu: `O(n)`, **không dùng phép chia**.

In [31]:
def exercise_check_straight_line(coordinates: List[List[int]]) -> bool:
    # TODO: lấy 2 điểm đầu làm A, B; mọi điểm C phải có cross(A, B, C) == 0
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_check_straight_line([[1,2],[2,3],[3,4],[4,5],[5,6],[6,7]]) == True
# assert exercise_check_straight_line([[1,1],[2,2],[3,4],[4,5],[5,6],[7,7]]) == False
# assert exercise_check_straight_line([[0,0],[0,1],[0,-1]]) == True

### Bài 6 — Mini challenge: phiên bản có **điểm trùng**

Giả sử input có thể chứa các điểm **trùng tọa độ** (như phiên bản rất cũ của bài 149). Điểm trùng vẫn được tính riêng: `[[1,1],[1,1],[2,2]]` → `3`. Viết `exercise_max_points_dup(points)`.

Chú ý: điểm trùng với điểm neo cho `(dy, dx) = (0, 0)` → `gcd = 0` → không được chia! Phải đếm riêng chúng, và chúng nằm trên **mọi** đường qua điểm neo.

In [32]:
def exercise_max_points_dup(points: List[List[int]]) -> int:
    # TODO: với mỗi điểm neo i: same = số j > i trùng tọa độ với i;
    #       các j còn lại đếm theo slope_key; ứng viên = same + 1 + max(counts, default 0)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_max_points_dup([[1,1],[1,1],[2,2],[2,2]]) == 4
# assert exercise_max_points_dup([[0,0],[0,0]]) == 2
# assert exercise_max_points_dup([[1,1],[1,1],[1,1]]) == 3
# assert exercise_max_points_dup([[1,1],[2,2],[1,1],[3,4]]) == 3
# assert exercise_max_points_dup([[1,1],[3,2],[5,3],[4,1],[2,3],[1,4]]) == 4

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: dùng đúng công thức `cross = dx1·dy2 - dy1·dx2` với `AB = (dx1, dy1)`, `AC = (dx2, dy2)`.
- Mức 2: câu (c) có `dx1 = 0` — công thức vẫn dùng được, không cần trường hợp đặc biệt.
- Mức 3: (a) `AB = (2,4)`, `AC = (3,6)` → `2·6 - 4·3 = ?`

**Bài 2**
- Mức 1: `math.gcd` luôn trả số không âm; `gcd(0, 0)` là trường hợp đặc biệt.
- Mức 2: với `slope_key`, chia gcd trước rồi mới nhìn dấu của `dx`.
- Mức 3: `slope_key(-6, -9)`: gcd = 3 → `(-2, -3)` → `dx < 0` nên đổi dấu → ?

**Bài 3**
- Mức 1: chỉ có 2 bước: rút gọn, rồi chuẩn hóa dấu.
- Mức 2: `g = math.gcd(dy, dx)`; `dy //= g`; `dx //= g`.
- Mức 3: `if dx < 0 or (dx == 0 and dy < 0): dy, dx = -dy, -dx`.

**Bài 4**
- Mức 1: hai vòng `for` lồng nhau: `i` là điểm neo, `j` chạy từ `i + 1`.
- Mức 2: `counts = {}` đặt **trong** vòng `i`; `best = max(best, max(counts.values(), default=0) + 1)`.
- Mức 3: đừng quên `if n <= 2: return n` ở đầu hàm, và `+ 1` cho chính điểm neo.

**Bài 5**
- Mức 1: hai điểm đầu xác định đường; các điểm còn lại phải nằm trên đường đó.
- Mức 2: dùng hàm `cross` ở mục 4; `all(...)` trả `True` khi mọi phần tử đều đúng.
- Mức 3: `return all(cross(A, B, C) == 0 for C in coordinates[2:])`.

**Bài 6**
- Mức 1: tách hai loại `j`: trùng điểm neo, và không trùng.
- Mức 2: `if dx == 0 and dy == 0: same += 1; continue` trước khi gọi `slope_key`.
- Mức 3: ứng viên của điểm neo `i` là `same + 1 + max(counts.values(), default=0)`; không cần `if n <= 2` riêng.

## 18. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1**

| Câu | AB | AC | cross | Thẳng hàng? |
|---|---|---|---:|---|
| (a) | `(2, 4)` | `(3, 6)` | `2·6 - 4·3 = 0` | Có |
| (b) | `(3, 1)` | `(6, 3)` | `3·3 - 1·6 = 3` | Không |
| (c) | `(0, 6)` | `(0, 101)` | `0·101 - 6·0 = 0` | Có (đường đứng `x = 2`) |
| (d) | `(2, 1)` | `(6, 3)` | `2·3 - 1·6 = 0` | Có |

**Bài 2**

| Dòng | Kết quả | Vì sao |
|---|---|---|
| `math.gcd(-12, 18)` | `6` | gcd luôn không âm |
| `math.gcd(0, 0)` | `0` | trường hợp đặc biệt (không dùng để chia được) |
| `slope_key(-6, -9)` | `(2, 3)` | gcd 3 → `(-2, -3)` → đổi dấu |
| `slope_key(5, -10)` | `(-1, 2)` | gcd 5 → `(1, -2)` → đổi dấu |
| `slope_key(0, -8)` | `(0, 1)` | đường ngang |
| `slope_key(-3, 0)` | `(1, 0)` | đường đứng, `dx = 0` và `dy < 0` → đổi dấu |

In [33]:
# Kiểm chứng Bài 1 và Bài 2
assert cross((0, 0), (2, 4), (3, 6)) == 0
assert cross((1, 2), (4, 3), (7, 5)) == 3
assert cross((2, -1), (2, 5), (2, 100)) == 0
assert cross((-1, -1), (1, 0), (5, 2)) == 0

assert math.gcd(-12, 18) == 6
assert math.gcd(0, 0) == 0
assert slope_key(-6, -9) == (2, 3)
assert slope_key(5, -10) == (-1, 2)
assert slope_key(0, -8) == (0, 1)
assert slope_key(-3, 0) == (1, 0)
print("Bài 1, Bài 2: PASS")

Bài 1, Bài 2: PASS


In [34]:
def answer_slope_key(dy: int, dx: int) -> tuple:
    g = math.gcd(dy, dx)
    dy //= g
    dx //= g
    if dx < 0 or (dx == 0 and dy < 0):
        dy, dx = -dy, -dx
    return (dy, dx)


def answer_max_points(points: List[List[int]]) -> int:
    n = len(points)
    if n <= 2:
        return n
    best = 1
    for i in range(n):
        counts = {}
        for j in range(i + 1, n):
            key = answer_slope_key(points[j][1] - points[i][1], points[j][0] - points[i][0])
            counts[key] = counts.get(key, 0) + 1
        best = max(best, max(counts.values(), default=0) + 1)
    return best


def answer_check_straight_line(coordinates: List[List[int]]) -> bool:
    A, B = coordinates[0], coordinates[1]
    return all(cross(A, B, C) == 0 for C in coordinates[2:])


def answer_max_points_dup(points: List[List[int]]) -> int:
    n = len(points)
    best = 0
    for i in range(n):
        same = 0
        counts = {}
        for j in range(i + 1, n):
            dy = points[j][1] - points[i][1]
            dx = points[j][0] - points[i][0]
            if dx == 0 and dy == 0:
                same += 1
                continue
            key = answer_slope_key(dy, dx)
            counts[key] = counts.get(key, 0) + 1
        best = max(best, same + 1 + max(counts.values(), default=0))
    return best


# Bài 3
assert answer_slope_key(4, 6) == (2, 3)
assert answer_slope_key(-4, -6) == (2, 3)
assert answer_slope_key(4, -6) == (-2, 3)
assert answer_slope_key(0, -5) == (0, 1)
assert answer_slope_key(-7, 0) == (1, 0)
# Bài 4
assert answer_max_points([[1, 1], [2, 2], [3, 3]]) == 3
assert answer_max_points([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]) == 4
assert answer_max_points([[0, 0]]) == 1
assert answer_max_points([[0, 0], [1, 1], [-1, -1], [5, 0]]) == 3
# Bài 5
assert answer_check_straight_line([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7]]) == True
assert answer_check_straight_line([[1, 1], [2, 2], [3, 4], [4, 5], [5, 6], [7, 7]]) == False
assert answer_check_straight_line([[0, 0], [0, 1], [0, -1]]) == True
# Bài 6
assert answer_max_points_dup([[1, 1], [1, 1], [2, 2], [2, 2]]) == 4
assert answer_max_points_dup([[0, 0], [0, 0]]) == 2
assert answer_max_points_dup([[1, 1], [1, 1], [1, 1]]) == 3
assert answer_max_points_dup([[1, 1], [2, 2], [1, 1], [3, 4]]) == 3
assert answer_max_points_dup([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]) == 4

# Đối chiếu: không có điểm trùng thì bản dup phải khớp bản thường
random.seed(2024)
for _ in range(300):
    pts = random.sample(grid, random.randint(1, 10))
    assert answer_max_points(pts) == answer_max_points_dup(pts) == max_points_bruteforce(pts)
print("Đáp án Bài 3–6: PASS")

Đáp án Bài 3–6: PASS


## 19. Template ngắn nên nhớ

```python
from math import gcd

class Solution:
    def maxPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        if n <= 2:
            return n
        best = 1
        for i in range(n):                      # điểm neo
            xi, yi = points[i]
            counts = {}                         # MỚI cho mỗi điểm neo
            for j in range(i + 1, n):
                dy, dx = points[j][1] - yi, points[j][0] - xi
                g = gcd(dy, dx)
                dy, dx = dy // g, dx // g       # rút gọn
                if dx < 0 or (dx == 0 and dy < 0):
                    dy, dx = -dy, -dx           # chuẩn hóa dấu
                counts[(dy, dx)] = counts.get((dy, dx), 0) + 1
            best = max(best, max(counts.values(), default=0) + 1)
        return best
```

Câu thần chú: **"Neo một điểm — hướng là phân số tối giản `(dy, dx)` — dấu chuẩn `dx > 0` — đếm bằng dict — cộng 1 cho điểm neo."** Và khi chỉ cần hỏi "3 điểm có thẳng hàng không" thì dùng **tích chéo bằng 0**.

## 20. Checklist trước khi submit

- Mình có **tránh float** (dùng `(dy, dx)` số nguyên / tích chéo / `Fraction`) chưa?
- Đường **thẳng đứng** (`dx = 0`) và **nằm ngang** (`dy = 0`) có ra đúng một key mỗi loại chưa?
- Đã **chia gcd** và **chuẩn hóa dấu** (`(1,-2)` và `(-1,2)` cùng key) chưa?
- `counts` có được **tạo mới cho mỗi điểm neo** (không gộp các đường song song) chưa?
- Đã **`+ 1`** cho chính điểm neo chưa? `max()` có `default=0` (hoặc chắc chắn dict không rỗng) chưa?
- Đã thử `n = 1`, `n = 2`, toàn bộ điểm thẳng hàng, đường đứng, điểm neo nằm giữa, hai example của đề chưa?